<a href="https://colab.research.google.com/github/jaylink-coder/miss-yaya/blob/main/yaya-ai/notebooks/train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jaylink-coder/miss-yaya/blob/main/yaya-ai/notebooks/train_colab.ipynb)

# Yaya-125M — Training Notebook

**Two modes:**

| Mode | When to use | Script |
|------|-------------|--------|
| **Consolidation** | ✅ Use this now — fixes catastrophic forgetting from the 16-phase run | `run_consolidation.py` |
| **Curriculum phases** | For future capability expansion (new phases) | `colab_run_phases.py` |

**Required:** Runtime → Change runtime type → **T4 GPU** (or better)

## Step 0 — Check GPU

In [29]:
import torch, os, sys

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print(f'GPU : {name}  ({vram} GB VRAM)')
    print(f'PyTorch: {torch.__version__}')
    dtype = 'bfloat16' if torch.cuda.is_bf16_supported() else 'float16'
    print(f'dtype: {dtype}')
else:
    raise RuntimeError('No GPU. Go to Runtime → Change runtime type → T4 GPU')

GPU : Tesla T4  (15.6 GB VRAM)
PyTorch: 2.10.0+cu128
dtype: bfloat16


## Step 1 — Mount Drive + clone repo

In [30]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.


In [31]:
import os, sys, subprocess

REPO = '/content/miss-yaya'
ROOT = f'{REPO}/yaya-ai'

if not os.path.exists(REPO):
    print('Cloning repo...')
    subprocess.run(['git', 'clone', 'https://github.com/jaylink-coder/miss-yaya.git', REPO], check=True)
else:
    print('Pulling latest...')
    result = subprocess.run(['git', 'pull'], cwd=REPO, capture_output=True, text=True)
    print(result.stdout.strip() or 'Already up to date.')

os.chdir(ROOT)
sys.path.insert(0, ROOT)
print(f'Ready: {os.getcwd()}')

Pulling latest...
Updating 5a92134..3f68c2e
Fast-forward
 yaya-ai/scripts/run_consolidation.py | 15 ++++++++++++---
 1 file changed, 12 insertions(+), 3 deletions(-)
Ready: /content/miss-yaya/yaya-ai


In [32]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'sentencepiece', 'pyyaml', 'huggingface_hub'], check=True)
print('Dependencies ready.')

Dependencies ready.


## Step 2 — Set HuggingFace token

Add `HF_TOKEN` in **Colab Secrets** (🔑 icon in left sidebar). Recommended over pasting here.

In [33]:
import os

hf_token = ''
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN') or ''
    if hf_token: print('HF token loaded from Colab Secrets.')
except Exception:
    pass

if not hf_token:
    hf_token = ''  # ← paste hf_xxx here if needed

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print('HF_TOKEN set.')
else:
    print('WARNING: No HF_TOKEN — Hub pull/push disabled.')

HF token loaded from Colab Secrets.
HF_TOKEN set.


## Step 3 — Consolidation Training ⭐ (run this now)

The 16-phase sequential run caused **catastrophic forgetting** — Kenya/Swahili dropped to 0%.

This fixes it by training on **all phase data mixed together** with smart weighting:
- Kenya/Swahili, Open Knowledge, Hard Reasoning → **3×** oversampled
- Instruction Following, Language → **2×** oversampled  
- Everything else → 1×

**19,868 examples · 3,000 steps · LR 5e-6 · ~2.5 hours on T4**

Expected result: model-only score jumps from **39% → 65%+**

In [ ]:
# Run consolidation — pulls phase16 checkpoint from Hub, trains, pushes result
!python scripts/run_consolidation.py

  Yaya — Consolidation Training
  Platform: Colab
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Scanning HF Hub for checkpoints...
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
curriculum-phase05-step02000/model.pt: 100% 516M/516M [00:05<00:00, 98.8MB/s]
metadata.json: 100% 62.0/62.0 [00:00<00:00, 251kB/s]
  No valid checkpoint found on Hub.
  Using local checkpoint: /content/checkpoints/yaya-125m-consolidation/curriculum-phase05-step02000/curriculum-phase05-step02000/model.pt

  Starting checkpoint: /content/checkpoints/yaya-125m-consolidation/curriculum-phase05-step02000/curriculum-phase05-step02000/model.pt
  Checkpoint loss: 0.2353
  Consolidation data: 19,868 examples
  GPU: TESLA T4 (14

## Step 4 — Benchmark after consolidation

Should see model-only Kenya go from 0% → 50%+, Swahili 0% → 60%+, Hard Reasoning 0% → 20%+

In [ ]:
import glob, os

# Look in consolidation dir first, then curriculum
search_paths = [
    '/content/checkpoints/yaya-125m-consolidation/**/*.pt',
    '/content/checkpoints/yaya-125m-curriculum/**/*.pt',
]
all_ckpts = []
for pattern in search_paths:
    all_ckpts.extend(glob.glob(pattern, recursive=True))

all_ckpts = sorted(all_ckpts, key=os.path.getmtime, reverse=True)
if all_ckpts:
    latest_dir = os.path.dirname(all_ckpts[0])
    print(f'Benchmarking: {latest_dir}')
    !python scripts/benchmark.py --checkpoint {latest_dir} --dual
else:
    print('No checkpoint found.')

## Step 5 — Test Yaya directly

Quick sanity check — ask her things she should know now.

In [ ]:
import torch, glob, os, sys
sys.path.insert(0, '/content/miss-yaya/yaya-ai')

from src.utils.config import load_model_config
from src.model.yaya_model import YayaForCausalLM
from src.tokenizer.tokenizer import YayaTokenizer, ASSISTANT_TOKEN
from src.inference.generator import TextGenerator, GenerationConfig

all_ckpts = sorted(
    glob.glob('/content/checkpoints/**/*.pt', recursive=True),
    key=os.path.getmtime, reverse=True
)
if not all_ckpts:
    print('No checkpoint. Run training first.')
else:
    ckpt_path = all_ckpts[0]
    print(f'Loading: {ckpt_path}')
    device = 'cuda'
    model  = YayaForCausalLM(load_model_config('configs/model/yaya_125m.yaml')).to(device)
    state  = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(state.get('model_state_dict', state))
    model.eval()

    tok = YayaTokenizer('data/tokenizer/yaya_tokenizer.model')
    gen = TextGenerator(model, tok, device=device)
    cfg = GenerationConfig(max_new_tokens=150, temperature=0.7, top_p=0.9, repetition_penalty=1.5)
    SYS = 'You are Yaya, a helpful, honest, and friendly AI assistant.'

    TESTS = [
        'What is the capital of Kenya?',
        'What is the Swahili word for water?',
        'What is 15% of 240?',
        'What is the largest lake in Africa?',
        'Hot is to cold as tall is to what?',
        'Who are you?',
    ]
    for q in TESTS:
        msgs   = [{'role':'system','content':SYS}, {'role':'user','content':q}]
        prompt = tok.format_chat(msgs) + '\n' + ASSISTANT_TOKEN + '\n'
        ans    = gen.generate(prompt, config=cfg).strip()
        print(f'Q: {q}')
        print(f'A: {ans[:120]}')
        print()

## Step 6 — Run more curriculum phases (optional)

After consolidation is working well, you can continue expanding capabilities.

In [ ]:
# Optional: continue curriculum from consolidated checkpoint
# !python scripts/colab_run_phases.py --phase 1-4 --no-clone

## Step 7 — Download checkpoint (optional)

In [ ]:
import glob, shutil, os
from google.colab import files

all_ckpts = sorted(
    glob.glob('/content/checkpoints/**/*.pt', recursive=True),
    key=os.path.getmtime, reverse=True
)
if all_ckpts:
    latest_dir = os.path.dirname(all_ckpts[0])
    shutil.make_archive('/content/yaya-trained', 'zip', latest_dir)
    print(f'Zipping {latest_dir}...')
    files.download('/content/yaya-trained.zip')
else:
    print('No checkpoint to download.')